# VLA Foundry Data Visualization

This notebook demonstrates how to visualize robotics datasets using VLA Foundry's visualization tools built on [Rerun](https://rerun.io/).

## Prerequisites

Before running this notebook, ensure you have:
1. Installed the VLA Foundry kernel: `bash tutorials/install_kernel.sh`
2. Selected the "Python (vla_foundry)" kernel in Jupyter

## Overview

The visualization pipeline:
1. Downloads a sample dataset (PickAndPlaceBox)
2. Runs the visualization script to generate Rerun visualization
3. Displays the interactive 3D viewer inline in the notebook

## Configuration

Set the task name and visualization parameters:

In [1]:
# Dataset configuration
TASK_NAME = "PickAndPlaceBox"
DATA_PATH = f"data/{TASK_NAME}"

# Visualization parameters
NUM_EPISODES = 3  # Number of episodes to visualize
SUBSAMPLE_FACTOR = 10  # Visualize every Nth sample (higher = faster, less data)

## Setup

Initialize the Rerun visualizer backend:

In [2]:
# Set environment variable to use the Rerun visualizer backend.
import os

os.environ["VISUALIZER"] = "rerun"

## Download Sample Dataset

Download the PickAndPlaceBox dataset for visualization:

In [3]:
!python ../scripts/download_dataset.py --task {TASK_NAME} --local_path {DATA_PATH}

Version: v0.1.0
Local path: /home/isabellahuang/TRI-ML/vla_foundry_internal/tutorials/data/PickAndPlaceBox

Fetching metadata for shards/
  Downloaded manifest.jsonl (2.4 KB)
  Downloaded stats.json (27.7 MB)
  Downloaded preprocessing_config.yaml (1.5 KB)
  Downloaded processing_metadata.json (3.3 KB)

Found 52 shards in manifest
Total sequences: 5175
  [1/52] 25.3 MB - shard_000003.tar
  [2/52] 50.5 MB - shard_000002.tar
  [3/52] 75.8 MB - shard_000004.tar
  [4/52] 101.1 MB - shard_000005.tar
  [5/52] 126.3 MB - shard_000000.tar
  [6/52] 151.6 MB - shard_000006.tar
  [7/52] 176.8 MB - shard_000007.tar
  [8/52] 202.1 MB - shard_000001.tar
  [9/52] 227.4 MB - shard_000008.tar
  [10/52] 252.7 MB - shard_000009.tar
  [11/52] 278.0 MB - shard_000010.tar
  [12/52] 303.3 MB - shard_000011.tar
  [13/52] 328.6 MB - shard_000012.tar
  [14/52] 353.9 MB - shard_000013.tar
  [15/52] 379.0 MB - shard_000014.tar
  [16/52] 404.3 MB - shard_000016.tar
  [17/52] 429.6 MB - shard_000015.tar
  [18/52] 4

## Visualize Dataset

The visualization uses the bash script wrapper to generate a Rerun visualization that displays inline in the notebook.

In [ ]:
# Build the visualization command
BASH_CMD = (
    f"bash examples/visualization/visualize_data.sh --ordered "
    f"--subsample={SUBSAMPLE_FACTOR} --num_episodes={NUM_EPISODES} {DATA_PATH}"
)

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

import rerun as rr

# Find project root by looking for pyproject.toml
current = Path.cwd()
while current != current.parent:
    if (current / "pyproject.toml").exists() and (current / "examples" / "visualization").exists():
        project_root = str(current)
        break
    current = current.parent
else:
    raise RuntimeError("Could not find project root (looking for pyproject.toml)")

# Get Python code from bash script with --print-command flag
cmd_parts = shlex.split(BASH_CMD)
cmd_parts.insert(2, "--print-command")
cmd_parts = [part.replace(DATA_PATH, f"tutorials/{DATA_PATH}") if DATA_PATH in part else part for part in cmd_parts]

try:
    result = subprocess.run(cmd_parts, capture_output=True, text=True, check=True, cwd=project_root)
except subprocess.CalledProcessError as e:
    print(f"Error running visualization script (exit code {e.returncode}):")
    print(f"STDERR: {e.stderr}")
    raise

# Execute the generated Python code and display the viewer
os.chdir(project_root)
exec(result.stdout)

rr.notebook_show()